# 03 · Fase B — Intervención preliminar: número de object queries
**Antes de ejecutar:** las hipótesis H1–H3 deben estar escritas y commiteadas en `DESIGN_LOG.md`.
Una hipótesis, un cambio (N), una medición. Mismo protocolo y semilla en todas las corridas.

In [ ]:
# Colab/Kaggle: clona el repo y instala dependencias (ajusta la URL)
# !git clone https://github.com/<usuario>/detr-query-saturation.git && cd detr-query-saturation
# !pip -q install -r requirements.txt
import sys; sys.path.append('../src' if __import__('os').path.exists('../src') else 'src')
from common import *
import torch, json, os, pandas as pd, matplotlib.pyplot as plt
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; SEED=42; set_seed(SEED); print(DEVICE)

In [ ]:
from transformers import DetrImageProcessor
CKPT='facebook/detr-resnet-50'
processor=DetrImageProcessor.from_pretrained(CKPT,size={'shortest_edge':512,'longest_edge':853})
IMG_TRAIN,IMG_VAL='<ruta/train/images>','<ruta/val/images>'   # ajusta
TRAIN_JSON,VAL_JSON='../data/train_sub.json','../data/val_sub.json'
EPOCHS,BS=8,2
groups,(q1,q2)=density_groups(VAL_JSON); print({k:len(v) for k,v in groups.items()},'cortes:',q1,q2)

In [ ]:
results={}
for N in (100,300):                         # añade 500 si hay tiempo
    set_seed(SEED)
    model=build_model(num_queries=N,num_labels=10,freeze_backbone=True)
    tl=make_loader(IMG_TRAIN,TRAIN_JSON,processor,bs=BS,shuffle=True)
    hist=train(model,tl,EPOCHS,device=DEVICE)
    res=evaluate_all(model,processor,IMG_VAL,VAL_JSON,groups=groups,device=DEVICE)
    results[N]={'loss':hist,'eval':res}; json.dump(results,open('../results/ablation_queries.json','w'),indent=1)
    print(N,res['overall']); del model; torch.cuda.empty_cache()

In [ ]:
rows=[{'N':N,'grupo':g,**m} for N,r in results.items() for g,m in r['eval'].items()]
T=pd.DataFrame(rows); print(T.round(3).to_string()); T.to_csv('../results/ablation_queries.csv',index=False)
P=T.pivot(index='grupo',columns='N',values='AR@500').loc[['low','medium','high','overall']]
P.plot(kind='bar'); plt.ylabel('AR@500'); plt.savefig('../results/ar500_por_densidad.png',dpi=150,bbox_inches='tight')

## Contraste con las hipótesis (completar en el design log)
- H1 (recall cae con N=100 en escenas densas): ¿se cumplió? ¿por qué?
- H2 (más queries ayuda sobre todo en escenas densas): ¿?
- H3 (costo bajo): mide latencia/FLOPs de cada N con el notebook 02.

⚠️ Entrenamiento corto en subconjunto: interpreten **diferencias relativas**, no el AP absoluto. Reporten esto como limitación.